In [ ]:
import cv2
import mediapipe as mp
import numpy as np
import time
from collections import deque

class FallDetector:
    def __init__(self):
        # Khởi tạo MediaPipe Pose
        self.mp_pose = mp.solutions.pose
        self.mp_drawing = mp.solutions.drawing_utils
        self.pose = self.mp_pose.Pose(
            min_detection_confidence=0.5,
            min_tracking_confidence=0.5
        )
        
        # Lưu lịch sử tỷ lệ chiều cao/chiều rộng
        self.aspect_ratio_history = deque(maxlen=10)
        self.velocity_history = deque(maxlen=5)
        
        # Trạng thái té ngã
        self.fall_detected = False
        self.fall_time = None
        self.alert_duration = 3  # Hiển thị cảnh báo trong 3 giây
        
    def calculate_aspect_ratio(self, landmarks, frame_shape):
        """Tính tỷ lệ chiều cao/chiều rộng của người"""
        h, w = frame_shape[:2]
        
        # Lấy tọa độ các điểm quan trọng
        nose = landmarks[self.mp_pose.PoseLandmark.NOSE.value]
        left_ankle = landmarks[self.mp_pose.PoseLandmark.LEFT_ANKLE.value]
        right_ankle = landmarks[self.mp_pose.PoseLandmark.RIGHT_ANKLE.value]
        left_shoulder = landmarks[self.mp_pose.PoseLandmark.LEFT_SHOULDER.value]
        right_shoulder = landmarks[self.mp_pose.PoseLandmark.RIGHT_SHOULDER.value]
        
        # Tính chiều cao (từ mũi đến mắt cá chân)
        ankle_y = max(left_ankle.y, right_ankle.y)
        height = abs(ankle_y - nose.y) * h
        
        # Tính chiều rộng (từ vai trái đến vai phải)
        width = abs(right_shoulder.x - left_shoulder.x) * w
        
        if width > 0:
            return height / width
        return 0
    
    def calculate_vertical_velocity(self, landmarks):
        """Tính vận tốc dọc của người"""
        # Lấy vị trí trung tâm cơ thể (hông)
        left_hip = landmarks[self.mp_pose.PoseLandmark.LEFT_HIP.value]
        right_hip = landmarks[self.mp_pose.PoseLandmark.RIGHT_HIP.value]
        center_y = (left_hip.y + right_hip.y) / 2
        return center_y
    
    def detect_fall(self, aspect_ratio, vertical_pos):
        """Phát hiện té ngã dựa trên tỷ lệ và vận tốc"""
        self.aspect_ratio_history.append(aspect_ratio)
        self.velocity_history.append(vertical_pos)
        
        # Kiểm tra nếu có đủ dữ liệu lịch sử
        if len(self.aspect_ratio_history) < 5 or len(self.velocity_history) < 3:
            return False
        
        # Điều kiện 1: Tỷ lệ chiều cao/chiều rộng đột ngột giảm (người nằm ngang)
        avg_ratio = np.mean(list(self.aspect_ratio_history)[-5:])
        is_horizontal = avg_ratio < 1.0  # Chiều rộng > chiều cao
        
        # Điều kiện 2: Có chuyển động nhanh xuống dưới
        velocity_list = list(self.velocity_history)
        velocity_change = velocity_list[-1] - velocity_list[0]
        is_falling_fast = velocity_change > 0.05  # Ngưỡng chuyển động
        
        # Phát hiện té ngã
        if is_horizontal and is_falling_fast:
            if not self.fall_detected:
                self.fall_detected = True
                self.fall_time = time.time()
                return True
        
        # Reset cảnh báo sau thời gian nhất định
        if self.fall_detected and (time.time() - self.fall_time > self.alert_duration):
            self.fall_detected = False
        
        return False
    
    def draw_info(self, frame, aspect_ratio, fall_status):
        """Vẽ thông tin lên frame"""
        h, w = frame.shape[:2]
        
        # Vẽ tỷ lệ chiều cao/chiều rộng
        cv2.putText(frame, f"Aspect Ratio: {aspect_ratio:.2f}", 
                    (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
        
        # Vẽ trạng thái
        if self.fall_detected:
            # Cảnh báo té ngã
            overlay = frame.copy()
            cv2.rectangle(overlay, (0, 0), (w, h), (0, 0, 255), -1)
            cv2.addWeighted(overlay, 0.3, frame, 0.7, 0, frame)
            
            cv2.putText(frame, "!!! TE NGA PHAT HIEN !!!", 
                       (w//2 - 250, h//2), cv2.FONT_HERSHEY_SIMPLEX, 
                       1.5, (0, 0, 255), 3)
            cv2.putText(frame, "FALL DETECTED!", 
                       (w//2 - 150, h//2 + 50), cv2.FONT_HERSHEY_SIMPLEX, 
                       1.2, (0, 0, 255), 3)
        else:
            cv2.putText(frame, "Trang thai: Binh thuong", 
                       (10, 60), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
    
    def process_frame(self, frame):
        """Xử lý một frame"""
        # Chuyển đổi BGR sang RGB
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        
        # Phát hiện pose
        results = self.pose.process(rgb_frame)
        
        fall_status = False
        aspect_ratio = 0
        
        if results.pose_landmarks:
            # Vẽ skeleton
            self.mp_drawing.draw_landmarks(
                frame, 
                results.pose_landmarks, 
                self.mp_pose.POSE_CONNECTIONS,
                self.mp_drawing.DrawingSpec(color=(0, 255, 0), thickness=2, circle_radius=2),
                self.mp_drawing.DrawingSpec(color=(0, 0, 255), thickness=2, circle_radius=2)
            )
            
            # Tính toán các chỉ số
            landmarks = results.pose_landmarks.landmark
            aspect_ratio = self.calculate_aspect_ratio(landmarks, frame.shape)
            vertical_pos = self.calculate_vertical_velocity(landmarks)
            
            # Phát hiện té ngã
            fall_status = self.detect_fall(aspect_ratio, vertical_pos)
        
        # Vẽ thông tin
        self.draw_info(frame, aspect_ratio, fall_status)
        
        return frame
    
    def run(self, video_source=0):
        """Chạy chương trình phát hiện té ngã"""
        cap = cv2.VideoCapture(video_source)
        
        print("Chuong trinh phat hien te nga dang chay...")
        print("Nhan 'q' de thoat")
        
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break
            
            # Xử lý frame
            frame = self.process_frame(frame)
            
            # Hiển thị
            cv2.imshow('Fall Detection System', frame)
            
            # Thoát khi nhấn 'q'
            if cv2.waitKey(1) & 0xFF == ord('q'):
                break
        
        cap.release()
        cv2.destroyAllWindows()

# Chạy chương trình
if __name__ == "__main__":
    detector = FallDetector()
    
    # Sử dụng webcam (0) hoặc đường dẫn video
    #detector.run(0)  # Webcam
    detector.run("video.mp4")  # Hoặc file video

Chuong trinh phat hien te nga dang chay...
Nhan 'q' de thoat
Chuong trinh phat hien te nga dang chay...
Nhan 'q' de thoat
